# 🚀 Train Improved Soccer Ball Detection Model

**Goal**: Train YOLO model with 7,335 labeled images to achieve 95% accuracy

**Dataset**: 
- 7,335 images with 6,935 ball annotations
- Nearly doubled from original 4,234 images
- Includes targeted YouTube soccer content

In [ ]:
# Install required packages
!pip install ultralytics roboflow

from ultralytics import YOLO
import os
from pathlib import Path

## 📁 Upload Dataset - UPDATED INSTRUCTIONS

**IMPORTANT: Use the FIXED dataset file!**

1. Upload `keepups_dataset_7335_FIXED.zip` (not the old one!)
2. This file is located at: `/Users/andreworozco/soccer app/training_data/keepups_dataset_7335_FIXED.zip`
3. Drag and drop the file into the Colab file browser on the left
4. Wait for upload to complete (587 MB will take 10-15 minutes)

**Why the FIXED version?**
- Original dataset had corrupted label files with literal `\n` characters
- Fixed version has proper YOLO format that won't cause training errors
- This prevents the timeout at epoch 87 that you experienced before

In [ ]:
# Extract dataset (adjust filename as needed)
!unzip -q keepups_dataset.zip
!ls -la keepups_dataset/

# Check dataset structure
!echo "Images:"
!ls keepups_dataset/images/ | wc -l
!echo "Labels:"
!ls keepups_dataset/labels/ | wc -l
!echo "Dataset config:"
!cat keepups_dataset/dataset.yaml

## 🔥 Train Model

Training with expanded dataset for improved accuracy

In [ ]:
# Fix dataset.yaml path for Colab
dataset_config = """
path: /content/keepups_dataset
train: images
val: images

names:
  0: ball

nc: 1
"""

with open('keepups_dataset/dataset.yaml', 'w') as f:
    f.write(dataset_config)

print("✅ Dataset config updated for Colab")

In [ ]:
# Load pretrained YOLOv8 model
model = YOLO('yolov8n.pt')

print("🚀 Starting training with 7,335 images...")
print("🎯 Target: 95% accuracy (up from 76%)")

# Train the model
results = model.train(
    data='keepups_dataset/dataset.yaml',
    epochs=100,          # More epochs for better convergence
    imgsz=640,          # Standard image size
    batch=16,           # Adjust based on GPU memory
    patience=20,        # Early stopping
    save=True,
    cache=True,         # Cache for faster training
    device=0,           # Use GPU
    workers=2,          # Colab workers
    project='runs/train',
    name='improved_soccer_ball',
    exist_ok=True
)

## 📊 Training Results

View training metrics and performance curves

In [ ]:
# Display training results
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Show training curves
results_path = 'runs/train/improved_soccer_ball/'

print("📈 Training Curves:")
display(Image(f'{results_path}/results.png'))

print("\n🔍 Validation Examples:")
display(Image(f'{results_path}/val_batch0_pred.png'))

print("\n📊 Confusion Matrix:")
display(Image(f'{results_path}/confusion_matrix.png'))

## 💾 Download Trained Model

Download the best model to use in your soccer app

In [ ]:
# Copy best model for download
import shutil

best_model_path = 'runs/train/improved_soccer_ball/weights/best.pt'
download_path = 'soccer_ball_improved.pt'

if os.path.exists(best_model_path):
    shutil.copy2(best_model_path, download_path)
    print(f"✅ Model ready for download: {download_path}")
    print(f"📊 File size: {os.path.getsize(download_path) / 1024 / 1024:.1f} MB")
    
    # Show final metrics
    print("\n🎯 Final Training Metrics:")
    if hasattr(results, 'results_dict'):
        metrics = results.results_dict
        print(f"mAP@0.5: {metrics.get('metrics/mAP50(B)', 'N/A')}")
        print(f"Precision: {metrics.get('metrics/precision(B)', 'N/A')}")
        print(f"Recall: {metrics.get('metrics/recall(B)', 'N/A')}")
else:
    print("❌ Best model not found")

print("\n📥 Download Instructions:")
print("1. Right-click on 'soccer_ball_improved.pt' in the file browser")
print("2. Select 'Download'")
print("3. Save to: /Users/andreworozco/soccer app/models/")
print("4. Rename to: soccer_ball_trained.pt (to replace old model)")

## 🧪 Quick Test

Test the model on a sample image to verify it's working

In [ ]:
# Load the trained model
trained_model = YOLO('soccer_ball_improved.pt')

# Test on a sample image from the dataset
import glob
sample_images = glob.glob('keepups_dataset/images/*.jpg')[:5]

print(f"🧪 Testing on {len(sample_images)} sample images...")

for img_path in sample_images:
    results = trained_model(img_path, conf=0.05)  # Low confidence for testing
    
    # Count detections
    detections = len(results[0].boxes) if results[0].boxes is not None else 0
    print(f"📸 {Path(img_path).name}: {detections} ball(s) detected")
    
    # Show first result with predictions
    if img_path == sample_images[0]:
        results[0].show()

print("\n✅ Model test complete! Ready for full evaluation.")